In [1]:
# Устрановка swisseph package
!pip install pyswisseph

# Установка geopy module
!pip install geopy

# Установка модуля поправок времени
!pip install pytz

# После выполнения установки можно продолжать основной код

In [2]:
# Импорт необходимых библиотек
import swisseph as swe
from datetime import datetime
from geopy.geocoders import Nominatim
from zoneinfo import ZoneInfo
from datetime import datetime, timedelta
import timezonefinder
import pytz

# Указываем путь к эфемеридам и файлу королевских звезд sefstars.txt
swe.set_ephe_path(r"C:\Users\demyd\Astro")

# Класс для расчета положения планет и куспидов домов
class NatalChart:

    def __init__(self, city_country, birth_datetime):
        """
        city_country: str, например "Paris, France"
        birth_datetime: datetime без tzinfo
        """
        self.city_country = city_country
        self.birth_datetime = birth_datetime

        # -------- 1. Получаем координаты --------
        geolocator = Nominatim(user_agent="astro_app")
        location = geolocator.geocode(city_country)

        if location is None:
            raise ValueError(f"Location '{city_country}' not found")

        self.latitude = location.latitude
        self.longitude = location.longitude

        # -------- 1.1 Топоцентрическая система координат --------
        swe.set_topo(self.longitude, self.latitude, 0)

        # -------- 2. Определяем часовой пояс --------
        tf = timezonefinder.TimezoneFinder()
        tz_str = tf.timezone_at(lng=self.longitude, lat=self.latitude)

        if tz_str is None:
            raise ValueError("Не удалось определить часовой пояс")

        self.timezone_str = tz_str

        # -------- 3. Юлианская дата (UTC) с ручным исключением для Oslo 1965,для карты Gill, Thomas --------
        if self.city_country.lower() == "oslo, norway" and self.birth_datetime.year == 1965:
            # Историческое летнее время в Oslo: UTC+2
            print("Applying manual DST correction for Oslo 1965")
            offset_hours = 2  # +2 для летнего времени
            local_dt = self.birth_datetime.replace(tzinfo=None)
            utc_dt = local_dt - timedelta(hours=offset_hours)
        # -------- Ручная поправка для Germany 1945-11-03 --------
        elif (
            "germany" in self.city_country.lower()
            and self.birth_datetime.year == 1945
            and self.birth_datetime.month == 11
            and self.birth_datetime.day == 3
        ):
            print("Applying manual DST correction for Germany 1945-11-03")
        
            offset_hours = 1  # astro.com фактически использует CET (UTC+1)
            local_dt = self.birth_datetime.replace(tzinfo=None)
            utc_dt = local_dt - timedelta(hours=offset_hours)
        # -------- Ручная поправка для Germany 1945 --------
        elif "germany" in self.city_country.lower() and self.birth_datetime.year == 1945:
            print("Applying manual DST correction for Germany 1945")
            offset_hours = 2
            local_dt = self.birth_datetime.replace(tzinfo=None)
            utc_dt = local_dt - timedelta(hours=offset_hours)
        else:
            tz = pytz.timezone(self.timezone_str)
            local_dt = tz.localize(self.birth_datetime, is_dst=None)
            utc_dt = local_dt.astimezone(pytz.utc)
        
        self.jd = swe.julday(
            utc_dt.year,
            utc_dt.month,
            utc_dt.day,
            utc_dt.hour + utc_dt.minute / 60 + utc_dt.second / 3600
        )

        # -------- 4. Дома и ASC/MC --------
        #удалить self.cusps, self.ascmc = swe.houses(self.jd, self.latitude, self.longitude)
        self.cusps, self.ascmc = swe.houses(self.jd, self.latitude, self.longitude, b'P')

        # Преобразуем cusps в список и добавим первый дом в конец
        self.cusps = list(self.cusps)

        if len(self.cusps) >= 12:
            self.cusps.append(self.cusps[1])

        # -------- 5. Планеты --------
        self.planets = self._calculate_planets()

        # -------- 6. Королевские звезды --------
        self.royal_stars = self._calculate_royal_stars()

        # -------- 7. Соединения --------
        self.star_conjunctions = self._calculate_star_conjunctions()

    # ================= HELPER =================

    def _format_zodiac(self, longitude):
    
        signs = [
            "Aries","Taurus","Gemini","Cancer","Leo","Virgo",
            "Libra","Scorpio","Sagittarius","Capricorn","Aquarius","Pisces"
        ]
    
        sign_index = int(longitude // 30) % 12
        sign_degree = longitude % 30
    
        deg = int(sign_degree)
        minutes = int((sign_degree - deg) * 60)
    
        return f"{deg}°{minutes:02d}′ {signs[sign_index]}"

    def _angular_diff(self, lon1, lon2):

        diff = abs(lon1 - lon2)
        return min(diff, 360 - diff)

    def _get_house(self, longitude):
        """
        Определяет, в каком доме находится планета.
        House 1 = ASC
        Houses 2-12 = cusps[1..12]
        """

        houses_boundaries = [self.ascmc[0]] + self.cusps[1:13]

        for i in range(12):

            start = houses_boundaries[i]
            end = houses_boundaries[(i + 1) % 12]

            if start < end:
                if start <= longitude < end:
                    return i + 1
            else:
                if longitude >= start or longitude < end:
                    return i + 1

        return None

    # ================= PLANETS =================

    def _calculate_planets(self):

        planet_ids = {
            "Sun": swe.SUN,
            "Moon": swe.MOON,
            "Mercury": swe.MERCURY,
            "Venus": swe.VENUS,
            "Mars": swe.MARS,
            "Jupiter": swe.JUPITER,
            "Saturn": swe.SATURN,
            "Uranus": swe.URANUS,
            "Neptune": swe.NEPTUNE,
            "Pluto": swe.PLUTO,
        }

        planets = {}

        for name, pid in planet_ids.items():

            pos = swe.calc_ut(self.jd, pid)[0][0]

            planets[name] = {
                "longitude": pos,
                "zodiac": self._format_zodiac(pos),
                "house": self._get_house(pos)
            }

        return planets

    # ----------------- Управления и экзальтации -----------------

    SIGN_RULERS = {
        "Aries": ["Mars"],
        "Taurus": ["Venus"],
        "Gemini": ["Mercury"],
        "Cancer": ["Moon"],
        "Leo": ["Sun"],
        "Virgo": ["Mercury"],
        "Libra": ["Venus"],
        "Scorpio": ["Pluto", "Mars"],
        "Sagittarius": ["Jupiter"],
        "Capricorn": ["Saturn"],
        "Aquarius": ["Uranus", "Saturn"],
        "Pisces": ["Neptune", "Jupiter"]
    }

    SIGN_EXALTATIONS = {
        "Sun": "Aries",
        "Moon": "Taurus",
        "Mercury": "Aquarius",
        "Venus": "Pisces",
        "Mars": "Capricorn",
        "Jupiter": "Cancer",
        "Saturn": "Libra",
        "Uranus": "Scorpio",
        "Neptune": "Aquarius",
        "Pluto": "Leo"
    }

    # ================= ФУНКЦИЯ ПОЛУЧЕНИЯ ЗНАКА ПЛАНЕТЫ =================
    def _get_sign(self, longitude):
        signs = [
            "Aries","Taurus","Gemini","Cancer","Leo","Virgo",
            "Libra","Scorpio","Sagittarius","Capricorn","Aquarius","Pisces"
        ]
        index = int(longitude // 30)
        return signs[index]

    # ================= ФУНКЦИЯ ПРОВЕРКИ СВЯЗИ ЧЕРЕЗ ДИСПОЗИТОР И ВЗАИМНОЙ РЕЦЕПЦИИ =================
    def planetary_reception(self, planet1, planet2):
    
        if planet1 not in self.planets or planet2 not in self.planets:
            return "No"
    
        lon1 = self.planets[planet1]["longitude"]
        lon2 = self.planets[planet2]["longitude"]
    
        if lon1 is None or lon2 is None:
            return "No"
    
        sign1 = self._get_sign(lon1)
        sign2 = self._get_sign(lon2)
    
        domicile_hits = []
        exalt_hits = []
    
        # planet1 в знаке planet2
        if planet2 in self.SIGN_RULERS.get(sign1, []):
            domicile_hits.append(f"{planet1} in {planet2} domicile")
    
        if self.SIGN_EXALTATIONS.get(planet2) == sign1:
            exalt_hits.append(f"{planet1} in {planet2} exaltation")
    
        # planet2 в знаке planet1
        if planet1 in self.SIGN_RULERS.get(sign2, []):
            domicile_hits.append(f"{planet2} in {planet1} domicile")
    
        if self.SIGN_EXALTATIONS.get(planet1) == sign2:
            exalt_hits.append(f"{planet2} in {planet1} exaltation")
    
        # -------- логика вывода --------
    
        total_hits = domicile_hits + exalt_hits
    
        if len(total_hits) == 0:
            return ""
    
        # если только одно совпадение — оставляем только domicile
        if len(total_hits) == 1:
            if domicile_hits:
                return domicile_hits[0]
            else:
                return ""
    
        # если два и больше — возвращаем все
        return ", ".join(total_hits)
    
    # ================= Проверка выраженности Марса  =================
    def get_sign_rulers(self, sign_name):
        """Возвращает список планет, которые управляют знаком sign_name"""
        rulers = {
            "Aries": ["Mars"],
            "Taurus": ["Venus"],
            "Gemini": ["Mercury"],
            "Cancer": ["Moon"],
            "Leo": ["Sun"],
            "Virgo": ["Mercury"],
            "Libra": ["Venus"],
            "Scorpio": ["Mars", "Pluto"],
            "Sagittarius": ["Jupiter"],
            "Capricorn": ["Saturn"],
            "Aquarius": ["Saturn", "Uranus"],
            "Pisces": ["Jupiter", "Neptune"]
        }
        return rulers.get(sign_name, [])
    
    def mars_dataframe_columns(self):
        """
        Возвращает три группы условий для Марса как список строк:
        
        Колонка 1: Mars в domicile или exaltation
        Колонка 2: Mars связан с 1 домом
        Колонка 3: Mars связан с 2, 5 или 10 домом
        """
        col1_list = []
        col2_list = []
        col3_list = []
    
        if "Mars" not in self.planets:
            return col1_list, col2_list, col3_list
    
        mars_lon = self.planets["Mars"]["longitude"]
        mars_house = self._get_house(mars_lon)
        mars_sign = self._get_sign(mars_lon % 360)
    
        # ----------------- Колонка 1: достоинства и слабости Марса -----------------
        # Обители и экзальтации Марса по традиционной астрологии:
        # Domicile: Aries, Scorpio; Exaltation: Capricorn
        # Detriment: Taurus, Libra # Fall: Cancer
        domicile_signs = ["Aries", "Scorpio"]
        exaltation_signs = ["Capricorn"]
        detriment_signs = ["Taurus", "Libra"]
        fall_signs = ["Cancer"]
    
        if mars_sign in domicile_signs:
            col1_list.append("Mars in domicile")
        if mars_sign in exaltation_signs:
            col1_list.append("Mars in exaltation")
        
        if mars_sign in detriment_signs:
            col1_list.append("Mars in detriment")
        
        if mars_sign in fall_signs:
            col1_list.append("Mars in fall")
    
        # ----------------- Колонка 2: 1-й дом -----------------
        if hasattr(self, "cusps") and len(self.cusps) > 0:
            first_house_lon = self.cusps[0]
            rulers_1 = self.get_sign_rulers(self._get_sign(first_house_lon))
    
            # Mars управитель 1 дома
            if "Mars" in rulers_1:
                col2_list.append("Mars ruler of 1st house")
                rulers_1 = [r for r in rulers_1 if r != "Mars"]
    
            # Mars в аспекте с управителем 1 дома
            for r in rulers_1:
                if self.check_aspect("Mars", r):
                    col2_list.append("Mars aspect with ruler of 1st house")
                    break
    
            # Mars в 1 доме и соединение с ASC ≤2°
            asc = self.cusps[0]
            diff = abs((mars_lon - asc) % 360)
            diff = diff if diff <= 180 else 360 - diff
            if mars_house == 1 and diff <= 2:
                col2_list.append("Mars in 1st house with ASC")
    
        # ----------------- Колонка 3: связь с 2,5,10 домами -----------------
        target_houses = [2,5,10]
        for h in target_houses:
            if mars_house == h:
                col3_list.append(f"Mars in {h} house")
            if hasattr(self, "cusps") and len(self.cusps) >= h:
                house_lon = self.cusps[h-1]
                rulers = self.get_sign_rulers(self._get_sign(house_lon))
                rulers = [r for r in rulers if r != "Mars"]
                for r in rulers:
                    if self.check_aspect("Mars", r):
                        col3_list.append(f"Mars aspect with ruler of {h} house")
                        break
    
        # ----------------- Убираем дубли -----------------
        col1_list = list(dict.fromkeys(col1_list))
        col2_list = list(dict.fromkeys(col2_list))
        col3_list = list(dict.fromkeys(col3_list))
    
        return col1_list, col2_list, col3_list
    
    # ================= Проверка Сатурна =================

    def saturn_dataframe_columns(self):
        """
        Возвращает три группы условий для Сатурна как список строк, без дубликатов
        """
        col1_list = []
        col2_list = []
        col3_list = []
    
        if "Saturn" not in self.planets:
            return col1_list, col2_list, col3_list
    
        saturn_lon = self.planets["Saturn"]["longitude"]
        saturn_house = self._get_house(saturn_lon)
        saturn_sign = self._get_sign(saturn_lon % 360)
    
        # ----------------- Колонка 1: достоинства Сатурна -----------------
        # Domicile: Capricorn, Aquarius         # Exaltation: Libra
        # Detriment: Cancer, Leo         # Fall: Aries
        
        domicile_signs = ["Capricorn", "Aquarius"]
        exaltation_signs = ["Libra"]
        detriment_signs = ["Cancer", "Leo"]
        fall_signs = ["Aries"]
        
        if saturn_sign in domicile_signs:
            col1_list.append("Saturn in domicile")
        
        if saturn_sign in exaltation_signs:
            col1_list.append("Saturn in exaltation")
        
        # if saturn_sign in detriment_signs:
            # col1_list.append("Saturn in detriment")
        
        # if saturn_sign in fall_signs:
            # col1_list.append("Saturn in fall")
        
        # ----------------- Колонка 2: 1-й дом -----------------
        if hasattr(self, "cusps") and len(self.cusps) > 0:
            first_house_lon = self.cusps[0]
            rulers_1 = self.get_sign_rulers(self._get_sign(first_house_lon))
    
            # Saturn управитель 1 дома
            if "Saturn" in rulers_1:
                col2_list.append("Saturn ruler of 1st house")
                # если Сатурн управляет 1 домом, не нужно дублировать аспект с самим собой
                rulers_1 = [r for r in rulers_1 if r != "Saturn"]
    
            # Saturn в аспекте с управителем 1 дома
            for r in rulers_1:
                if self.check_aspect("Saturn", r):
                    col2_list.append(f"Saturn aspect with ruler of 1st house")
                    break
    
            # Saturn в 1 доме и соединение с ASC ≤2°
            asc = self.cusps[0]
            diff = abs((saturn_lon - asc) % 360)
            diff = diff if diff <= 180 else 360 - diff
            if saturn_house == 1 and diff <= 2:
                col2_list.append("Saturn in 1st house with ASC")
    
        # ----------------- Колонка 3: связь с 2,5,10 домами -----------------
        target_houses = [2,5,10]
        for h in target_houses:
            if saturn_house == h:
                col3_list.append(f"Saturn in {h} house")
            if hasattr(self, "cusps") and len(self.cusps) >= h:
                house_lon = self.cusps[h-1]
                rulers = self.get_sign_rulers(self._get_sign(house_lon))
                # если Сатурн управляет дом, не нужно дублировать аспект с самим собой
                rulers = [r for r in rulers if r != "Saturn"]
                for r in rulers:
                    if self.check_aspect("Saturn", r):
                        col3_list.append(f"Saturn aspect with ruler of {h} house")
                        break
    
        # ----------------- Убираем дубли -----------------
        col1_list = list(dict.fromkeys(col1_list))
        col2_list = list(dict.fromkeys(col2_list))
        col3_list = list(dict.fromkeys(col3_list))
    
        return col1_list, col2_list, col3_list
      

    # ================= Проверка, выражены ли Близнецы, Стрельцы, Рыбы и Водолей =================
    
    def check_ruler_in_specific_signs(self):
        """
        Проверяет 1-й дом:
          - стоит ли куспид дома в целевых знаках (Gemini, Sagittarius, Pisces, Aquarius)
          - находится ли управитель дома в этих знаках (по реальному положению планеты через _get_house)
        
        Возвращает список индикаторов или ["No"], если ничего нет.
        """
        indicators = []
        target_signs = ["Gemini", "Sagittarius", "Pisces", "Aquarius"]
    
        if not hasattr(self, "cusps") or not self.cusps or not hasattr(self, "planets"):
            return ["No"], sign_counts
    
        # Дома, которые проверяем
        houses_to_check = {1: "1st", 2: "2nd", 5: "5th", 10: "10th"}
    
        for house_num, house_name in houses_to_check.items():
            if len(self.cusps) < house_num:
                continue
            cusp_long = self.cusps[house_num - 1]  # исправлено
            cusp_sign = self._get_sign(cusp_long % 360)
        
            if cusp_sign in target_signs:
                indicators.append(f"{house_name} house cusp in {cusp_sign}")
        
            rulers = self.get_sign_rulers(cusp_sign)
            for ruler in rulers:
                ruler_planet = self.planets.get(ruler)
                if ruler_planet and "longitude" in ruler_planet:
                    ruler_long = ruler_planet["longitude"]
                    ruler_sign = self._get_sign(ruler_long % 360)
                    ruler_house = self._get_house(ruler_long)
                    if ruler_sign in target_signs:
                        indicators.append(
                            f"{house_name} house ruler in {ruler_sign}"
                        )
        
        # удаляем дубликаты индикаторов
        indicators = list(dict.fromkeys(indicators)) or [""]
    
        # считаем количество каждого знака по итоговому списку индикаторов
        sign_counts = {sign: 0 for sign in target_signs}
        for item in indicators:
            for sign in target_signs:
                if sign in item:
                    sign_counts[sign] += 1
    
        return indicators, sign_counts
    
    # ================= Проверка связей домов =================
    def check_house_pairs_connections(self):
        """
        Проверяет связи между парами домов:
        1-10, 2-5, 2-10, 5-10
        
        Связи:
          - Управители домов в аспекте
          - Управитель одного дома находится в другом доме
          - Аспект между планетой одного дома и управителем другого дома
          - Аспект между планетой одного дома и планетой другого дома
    
        Возвращает словарь: { "1-10": [список индикаторов], ... }
        """
        house_pairs = [(1, 10), (2, 5), (2, 10), (5, 10)]
        connections = {}
    
        for h1, h2 in house_pairs:
            indicators = []
    
            # проверяем, есть ли куспиды
            if len(self.cusps) < max(h1, h2):
                connections[f"{h1}-{h2}"] = [""]
                continue
    
            cusp1 = self.cusps[h1 - 1]
            cusp2 = self.cusps[h2 - 1]
            sign1 = self._get_sign(cusp1 % 360)
            sign2 = self._get_sign(cusp2 % 360)
    
            rulers1 = self.get_sign_rulers(sign1)
            rulers2 = self.get_sign_rulers(sign2)

            # ----------------- Общий управитель домов -----------------
            for r1 in rulers1:
                if r1 in rulers2:
                    indicators.append(f"{r1}, ruler of both {h1} and {h2}")
            
                    if r1 in self.planets:
                        ruler_house = self._get_house(self.planets[r1]["longitude"])
            
                        if ruler_house == h1:
                            indicators.append(f"{r1} ruler of both {h1} and {h2} located in {h1} house")
            
                        if ruler_house == h2:
                            indicators.append(f"{r1} ruler of both {h1} and {h2} located in {h2} house")
    
            # ----------------- 1. Управители домов в аспекте -----------------
            for r1 in rulers1:
                for r2 in rulers2:
                    if r1 != r2 and self.check_aspect(r1, r2):
                        indicators.append(f"{r1} ruler of {h1} in aspect with {r2} ruler of {h2}")
    
            # ----------------- 2. Управитель одного дома в другом доме -----------------
            for r1 in rulers1:
                if r1 not in rulers2 and r1 in self.planets and self._get_house(self.planets[r1]["longitude"]) == h2:
                    indicators.append(f"{r1} ruler of {h1} in {h2} house")
            
            for r2 in rulers2:
                if r2 not in rulers1 and r2 in self.planets and self._get_house(self.planets[r2]["longitude"]) == h1:
                    indicators.append(f"{r2} ruler of {h2} in {h1} house")
    
            # ----------------- 3. Аспект между планетой одного дома и управителем другого дома -----------------
            for planet_name, planet_data in self.planets.items():
                planet_house = self._get_house(planet_data["longitude"])
                # планета в h1 и управитель h2
                for r2 in rulers2:
                    if planet_house == h1 and planet_name != r2 and self.check_aspect(planet_name, r2):
                        indicators.append(f"{planet_name} planet of {h1} house in aspect with {r2} ruler of {h2} house")
            
                # планета в h2 и управитель h1
                for r1 in rulers1:
                    if planet_house == h2 and planet_name != r1 and self.check_aspect(planet_name, r1):
                        indicators.append(f"{planet_name} planet of {h2} house in aspect with {r1} ruler of {h1} house")                       
   
            # ----------------- 4. Аспект между планетой одного дома и планетой другого дома -----------------
            for p1_name, p1_data in self.planets.items():
                house1 = self._get_house(p1_data["longitude"])
                if house1 != h1 and house1 != h2:
                    continue
                for p2_name, p2_data in self.planets.items():
                    house2 = self._get_house(p2_data["longitude"])
                    if house2 != h1 and house2 != h2:
                        continue
                    if p1_name < p2_name and house1 != house2 and self.check_aspect(p1_name, p2_name):
                        indicators.append(f"{p1_name} planet of {house1} house in aspect with {p2_name} planet of {house2} house")
                
            # удаляем дубликаты
            indicators = list(dict.fromkeys(indicators)) or [""]
            connections[f"{h1}-{h2}"] = indicators
    
        return connections
    
    
    # ================= ROYAL STARS =================

    def _calculate_royal_stars(self):

        #stars_to_find = ["Aldebaran", "Regulus", "Antares", "Fomalhaut"]
        stars_to_find = [
            "Aldebaran", "Regulus", "Antares", "Fomalhaut", # королевские звезды (изначальная гипотеза о Стражах Неба)
            "Sirius", "Algol", "Rigel", "Bellatrix", "Capella",
            "Betelgeuse", "Canopus", "Castor", "Pollux",
            "Acrux", "Unukalhai", "Agena", "Toliman", "Vega",
            "Altair", "Achernar", "Procyon", "Dubhe", "Alphard", "Spica", "Arcturus"
        ]

        royal = {}

        for star in stars_to_find:

            try:

                result = swe.fixstar2_ut(star, self.jd)
                longitude = result[0][0]

                royal[star] = {
                    "longitude": longitude,
                    "zodiac": self._format_zodiac(longitude),
                    "house": self._get_house(longitude)
                }

            except Exception as e:

                royal[star] = {
                    "longitude": None,
                    "zodiac": None,
                    "house": None
                }

                print(f"Не удалось рассчитать {star}: {e}")

        return royal

    # ================= CONJUNCTIONS WITH ROYAL STARS =================

    def _calculate_star_conjunctions(self, orb=1.0):

        conjunctions = []

        if not hasattr(self, "planets") or not hasattr(self, "royal_stars"):
            return conjunctions

        for planet, pdata in self.planets.items():

            for star, sdata in self.royal_stars.items():

                if pdata["longitude"] is not None and sdata["longitude"] is not None:

                    diff = self._angular_diff(
                        pdata["longitude"],
                        sdata["longitude"]
                    )

                    if diff <= orb:

                        conjunctions.append({
                            "planet": planet,
                            "star": star,
                            "orb": round(diff, 2)
                        })

        return conjunctions

    # ================= снова звезды =================

    def star_conjunctions_verbose(self, orb=1.0):
        """
        Возвращает список соединений королевских звёзд с планетами.
        Для каждой планеты указывает:
          - дом, в котором находится планета
          - чем она управляет в карте (например, 1st house)
          - орб в формате градусов и минут (0°08′)
        """
        def format_orb(deg_float):
            deg = int(deg_float)
            minutes = int(round((deg_float - deg) * 60))
            return f"{deg}°{minutes:02d}′"
    
        if not hasattr(self, "planets") or not hasattr(self, "royal_stars") or not hasattr(self, "cusps"):
            return []
    
        # словарь знаков → управители
        default_rulers = {
            "Aries": ["Mars"],
            "Taurus": ["Venus"],
            "Gemini": ["Mercury"],
            "Cancer": ["Moon"],
            "Leo": ["Sun"],
            "Virgo": ["Mercury"],
            "Libra": ["Venus"],
            "Scorpio": ["Mars","Pluto"],
            "Sagittarius": ["Jupiter"],
            "Capricorn": ["Saturn"],
            "Aquarius": ["Saturn","Uranus"],
            "Pisces": ["Jupiter","Neptune"]
        }
    
        # список домов
        house_numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
    
        output = []
    
        for planet, pdata in self.planets.items():
            if pdata.get("longitude") is None:
                continue
    
            house = self._get_house(pdata["longitude"])
            sign = self._get_sign(pdata["longitude"] % 360)
    
            # управители знака
            if hasattr(self, "get_sign_rulers"):
                rulers = self.get_sign_rulers(sign)
            else:
                rulers = default_rulers.get(sign, [])
    
            # определяем, какие дома управляет эта планета
            houses_controlled = []
            for h in house_numbers:
                if h <= len(self.cusps):
                    cusp_sign = self._get_sign(self.cusps[h-1])
                    cusp_rulers = self.get_sign_rulers(cusp_sign) if hasattr(self, "get_sign_rulers") else default_rulers.get(cusp_sign, [])
                    if planet in cusp_rulers:
                        houses_controlled.append(f"{h}th house" if h != 1 else "1st house")
    
            ruler_str = ", ".join(houses_controlled) if houses_controlled else "none"
    
            for star, sdata in self.royal_stars.items():
                if sdata.get("longitude") is None:
                    continue
    
                diff = self._angular_diff(pdata["longitude"], sdata["longitude"])
                if diff <= orb:
                    orb_str = format_orb(diff)
                    output.append(
                        f"{planet} (house {house}, ruler of {ruler_str}) conjunct {star} (orb {orb_str})"
                    )
    
        return output

    # ================= УНИВЕРСАЛЬНАЯ ФУНКЦИЯ ДЛЯ АСПЕКТОВ =================

    def check_aspect(self, planet1, planet2, orb=6.0):
        """
        Универсальная проверка мажорных аспектов между двумя планетами
        planet1, planet2 — названия планет из self.planets
        orb — орбис
        """
    
        aspects = []
    
        if planet1 not in self.planets or planet2 not in self.planets:
            return aspects
    
        lon1 = self.planets[planet1]["longitude"]
        lon2 = self.planets[planet2]["longitude"]
    
        if lon1 is None or lon2 is None:
            return aspects
    
        major_angles = {
            "Conjunction": 0,
            "Opposition": 180,
            "Square": 90,
            "Trine": 120,
            "Sextile": 60
        }
    
        diff = self._angular_diff(lon1, lon2)
    
        for aspect_name, angle in major_angles.items():
    
            if abs(diff - angle) <= orb:
    
                aspects.append({
                    "aspect": aspect_name,
                    "orb": round(diff - angle, 2)
                })
    
        return aspects
    
    # ================= ФУНКЦИЯ ПЕРЕВОДА ДЕСЯТИЧНЫХ ГРАДУСОВ В МИНУТЫ И СЕКУНДЫ =================
        
    def _deg_to_degmin(self, value):
        """
        Переводит десятичные градусы в формат: 3°22′
        """
        value = abs(value)
        degrees = int(value)
        minutes = int(round((value - degrees) * 60))
    
        # защита если округление дало 60 минут
        if minutes == 60:
            degrees += 1
            minutes = 0
    
        return f"{degrees}°{minutes:02d}′"
    # ================= REPORT =================

    def print_report(self):

        print(f"\nNatal Chart Report for {self.city_country}")
        print("=" * 45)

        print(f"Latitude : {self.latitude}")
        print(f"Longitude: {self.longitude}")
        print(f"Timezone: {self.timezone_str}")

        print(f"\nAscendant : {self._format_zodiac(self.ascmc[0])}")
        print(f"Midheaven : {self._format_zodiac(self.ascmc[1])}")

        print("\nPlanets:")

        for planet, data in self.planets.items():
            print(f"{planet:<8}: {data['zodiac']} (House {data['house']})")

        print("\nRoyal Stars:")

        for star, data in self.royal_stars.items():

            if data["longitude"] is not None:
                print(f"{star:<10}: {data['zodiac']} (House {data['house']})")
            else:
                print(f"{star:<10}: Not found")

        print("\nConjunctions with Royal Stars (orb ≤ 1°):")

        if self.star_conjunctions:

            for c in self.star_conjunctions:
                orb_str = self._deg_to_degmin(c['orb'])
                print(f"{c['planet']} conjunct {c['star']} (orb {orb_str})")

        else:
            print("No conjunctions found")

        print("\nHouse Cusps:")

        print(f"House 1 (ASC) : {self._format_zodiac(self.ascmc[0])}")

        for i in range(2, 13):
            print(f"House {i:>2} : {self._format_zodiac(self.cusps[i-1])}")

In [3]:
# Тест для проверки работы класса NatalChart:
birth_dt = datetime(1965, 5, 16, 22, 15)
chart = NatalChart("Oslo, Norway", birth_dt)
chart.print_report()

Applying manual DST correction for Oslo 1965

Natal Chart Report for Oslo, Norway
Latitude : 59.9133301
Longitude: 10.7389701
Timezone: Europe/Oslo

Ascendant : 1°04′ Sagittarius
Midheaven : 9°35′ Libra

Planets:
Sun     : 25°43′ Taurus (House 6)
Moon    : 11°13′ Sagittarius (House 1)
Mercury : 1°47′ Taurus (House 4)
Venus   : 4°49′ Gemini (House 7)
Mars    : 12°40′ Virgo (House 9)
Jupiter : 5°31′ Gemini (House 7)
Saturn  : 15°46′ Pisces (House 3)
Uranus  : 10°41′ Virgo (House 9)
Neptune : 18°26′ Scorpio (House 11)
Pluto   : 13°41′ Virgo (House 9)

Royal Stars:
Aldebaran : 9°17′ Gemini (House 7)
Regulus   : 29°20′ Leo (House 8)
Antares   : 9°16′ Sagittarius (House 1)
Fomalhaut : 3°22′ Pisces (House 3)
Sirius    : 13°35′ Cancer (House 8)
Algol     : 25°40′ Taurus (House 6)
Rigel     : 16°20′ Gemini (House 7)
Bellatrix : 20°27′ Gemini (House 7)
Capella   : 21°21′ Gemini (House 7)
Betelgeuse: 28°15′ Gemini (House 7)
Canopus   : 14°27′ Cancer (House 8)
Castor    : 19°44′ Cancer (House 8)
P

In [4]:
# Скрипт для расчетов по данным в таблице с данными
import csv
from datetime import datetime

# Гипотезы 1-5, проверка аспектов
aspect_rules = [
    ("Mercury","Mars",6),
    ("Mercury","Uranus",6),
    ("Mercury","Saturn",6),
    ("Mercury","Pluto",6),
    ("Mars","Uranus",6),
    ("Mars","Pluto",6),
    ("Mars","Moon",7),
]

# Гипотезы 1-5, проверка взаимной рецепции
reception_pairs = [
    ("Mercury","Mars"),
    ("Mercury","Uranus"),
    ("Mercury","Saturn"),
    ("Mercury","Pluto"),
    ("Mars","Uranus"),
    ("Mars","Pluto"),
    ("Mars","Moon"),
]

def run_from_table(csv_file, results_file="results.csv"):
    """
    Прогоняет таблицу с данными по NatalChart и сохраняет результат в CSV.
    
    csv_file: входной CSV с колонками name, city, date, time, Sun, Moon, Ascendant
    results_file: выходной CSV с колонками:
        name, city, date, time, Mercury-Uranus Aspect, Проверка
    """

    results = []

    def deg_string_to_float(s):
        """
        Преобразует строки вида '12°19′ Sagittarius' или '03°33\' Aquarius' в градусы 0..360
        """
        s = s.replace("′", "'").replace("’", "'").replace("‘", "'").strip()
        parts = s.split()
        if len(parts) < 2:
            return None
        deg_min = parts[0].replace("°", " ").replace("'", " ").split()
        deg = int(deg_min[0])
        minute = int(deg_min[1]) if len(deg_min) > 1 else 0
        zodiac_name = parts[1].capitalize()
        zodiac_order = [
            "Aries", "Taurus", "Gemini", "Cancer", "Leo", "Virgo",
            "Libra", "Scorpio", "Sagittarius", "Capricorn", "Aquarius", "Pisces"
        ]
        if zodiac_name not in zodiac_order:
            return None
        zodiac_index = zodiac_order.index(zodiac_name)
        total_deg = zodiac_index*30 + deg + minute/60
        return total_deg

    with open(csv_file, newline='', encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            try:
                birth_dt = datetime.strptime(
                    row["date"] + " " + row["time"],
                    "%Y-%m-%d %H:%M"
                )

                print("\n" + "=" * 60)
                print("Processing:", row["name"])
                chart = NatalChart(row["city"], birth_dt)
                chart.print_report()

                # ----------------- Проверка гипотез 1-5 / аспекты / -----------------

                aspect_results = {}

                for p1, p2, orb in aspect_rules:
                
                    aspects = chart.check_aspect(p1, p2, orb)
                
                    if aspects:
                
                        def orb_to_deg_min(orb_val):
                            orb_val = abs(orb_val)
                            deg = int(orb_val)
                            minutes = int(round((orb_val - deg) * 60))
                            return f"{deg}°{minutes:02d}′"
                
                        aspect_str = ", ".join(
                            f"{a['aspect']} (orb {orb_to_deg_min(a['orb'])})"
                            for a in aspects
                        )
                
                    else:
                        aspect_str = ""
                
                    aspect_results[f"{p1}-{p2}"] = aspect_str

                # ----------------- Проверка гипотез 1-5 / управление через диспозитор или взаимная рецепция /  -----------------
                reception_results = {}

                for p1, p2 in reception_pairs:
                
                    reception = chart.planetary_reception(p1, p2)
                
                    col_name = f"{p1}-{p2} Dispositorship/Reception"
                
                    reception_results[col_name] = reception

                # ----------------- Проверка выраженности Марса  -----------------
                col1_mars, col2_mars, col3_mars = chart.mars_dataframe_columns()

                # ----------------- Проверка Сатурна -----------------
                col1_saturn, col2_saturn, col3_saturn = chart.saturn_dataframe_columns()

                # ----------------- Проверка выраженности Близнецов, Стрельцов, Рыб, Водолея -----------------
                indicators, sign_counts = chart.check_ruler_in_specific_signs()

                # ----------------- Проверка связей между парами домов -----------------
                house_connections = chart.check_house_pairs_connections()
                               
                # ----------------- Соединения с королевскими звёздами -----------------
                if chart.star_conjunctions:
                    royal_str = ", ".join(
                        f"{c['planet']} conjunct {c['star']} (orb {abs(c['orb']):.2f}°)"
                        for c in chart.star_conjunctions
                    )
                else:
                    royal_str = "No"

                # получаем список соединений с подробностями для звезд
                royal_verbose_list = chart.star_conjunctions_verbose(orb=1.0)

                # ----------------- Проверка Sun, Moon, Ascendant, совпадает ли расчет с данными astro.com -----------------
                check_items = {
                    "Sun": row.get("Sun", ""),
                    "Moon": row.get("Moon", ""),
                    "Ascendant": row.get("Ascendant", "")
                }

                check_result = "OK"
                for body, val in check_items.items():
                    chart_deg = chart.planets[body]["longitude"] if body != "Ascendant" else chart.ascmc[0]
                
                    if val.strip() == "" and chart_deg is None:
                        continue  # пусто в CSV и пусто в chart → OK
                
                    csv_deg = deg_string_to_float(val) if val.strip() != "" else None
                
                    if (csv_deg is None and chart_deg is not None) or (csv_deg is not None and chart_deg is None):
                        check_result = "Mismatch"
                        break
                
                    if csv_deg is not None and chart_deg is not None:
                        diff = abs((csv_deg - chart_deg + 180) % 360 - 180)
                        if diff > 1.0:
                            check_result = "Mismatch"
                            break

                # ----------------- Сохраняем результат -----------------

                result_row = {
                    "name": row["name"],
                    "city": row["city"],
                    "date": row["date"],
                    "time": row["time"],
                    "group": row["group"]
                }
                
                # функция для очистки кода от квадратных скобок и пустых значений в списках
                def list_to_cell(lst):
                    cleaned = [x for x in lst if str(x).strip() != ""]
                    return ", ".join(cleaned) if cleaned else ""
                
                # добавляем аспекты
                result_row.update(aspect_results)

                # добавляем взаимную рецепцию
                result_row.update(reception_results)

                # добавляем проверку Марса - 2-й вариант
                result_row["Mars dignity +/-"] = list_to_cell(col1_mars)
                result_row["Mars 1st House Relation"] = list_to_cell(col2_mars)
                result_row["Mars 2-5-10 House Relation"] = list_to_cell(col3_mars)

                # Добавляем в результирующую строку 2-й вариант по Сатурну
                result_row["Saturn dignity +"] = list_to_cell(col1_saturn)
                result_row["Saturn 1st House Relation"] = list_to_cell(col2_saturn)
                result_row["Saturn 2-5-10 House Relation"] = list_to_cell(col3_saturn)

                # добавляем проверку выраженности Близнецов, Стрельцов, Рыб 
                result_row["Specific Signs Asc/Asc Ruler"] = list_to_cell(indicators)

                # сохраняем количество для каждого знака в отдельные колонки
                for sign in ["Gemini", "Sagittarius", "Pisces", "Aquarius"]:
                    result_row[sign] = sign_counts.get(sign, 0)

                # сохраняем каждый список индикаторов для проверки домов в отдельные колонки
                for pair in ["1-10", "2-5", "2-10", "5-10"]:
                    result_row[f"House {pair} Connections"] = house_connections.get(pair, ["No"])
                    

                # ----------------- Проверка количества связей домов -----------------

                total_connections = 0

                for pair in ["1-10", "2-5", "2-10", "5-10"]:
                    col = f"House {pair} Connections"
                
                    indicators = house_connections.get(pair, [])
                
                    result_row[col] = list_to_cell(indicators)
                
                    count = len([x for x in indicators if str(x).strip() != ""])
                    result_row[f"House {pair} Count"] = count
                
                    total_connections += count
                
                result_row["Total House Connections"] = total_connections
                
                # добавляем остальные поля
                result_row["Star Conjunctions"] = list_to_cell(royal_verbose_list)
                
                # списки звезд
                royal_stars = {"Aldebaran", "Regulus", "Antares", "Fomalhaut"}
                bellatrix_star = "Bellatrix"
                
                royal_star_names = []
                bellatrix_names = []
                other_star_names = []
                
                for item in royal_verbose_list:
                    if "conjunct" in item:
                        star = item.split("conjunct")[1].split("(")[0].strip()
                
                        if star in royal_stars:
                            royal_star_names.append(star)
                
                        elif star == bellatrix_star:
                            bellatrix_names.append(star)
                
                        else:
                            other_star_names.append(star)
                
                # записываем в таблицу
                result_row["Royal Star Name"] = list_to_cell(royal_star_names)
                result_row["Bellatrix Star"] = list_to_cell(bellatrix_names)
                result_row["Other Stars"] = list_to_cell(other_star_names)
                                
                # сверка данных по Asc, Sun, Moon с astro.com с расчетными данными скрипта
                result_row["Check Asc, Sun, Moon"] = check_result
                
                # добавляем в список результатов
                results.append(result_row)

            except Exception as e:
                print("Ошибка для строки:", row)
                print(e)
             

    # ----------------- Записываем результаты в CSV -----------------
    if results:
    
        fieldnames = ["name", "city", "date", "time", "group"]
    
        # добавляем аспекты и рецепции рядом
        for p1, p2, _ in aspect_rules:
            fieldnames.append(f"{p1}-{p2}")
            fieldnames.append(f"{p1}-{p2} Dispositorship/Reception")

        # ----------------- добавляем колонки для Марса -----------------
        fieldnames += [
            "Mars dignity +/-",         # колонка 1
            "Mars 1st House Relation",          # колонка 2
            "Mars 2-5-10 House Relation"        # колонка 3
        ]

        # ----------------- добавляем колонки для Сатурна  -----------------
        fieldnames += [
            "Saturn dignity +",       # колонка 1
            "Saturn 1st House Relation",        # колонка 2
            "Saturn 2-5-10 House Relation"      # колонка 3
        ]

        # добавляем колонку для выраженных знаков
        fieldnames += ["Specific Signs Asc/Asc Ruler"]
        # добавляем колонки для количества каждого знака
        fieldnames += ["Gemini", "Sagittarius", "Pisces", "Aquarius"]

        # добавляем колонки для связей домов
        fieldnames += [
            "House 1-10 Connections",
            "House 2-5 Connections",
            "House 2-10 Connections",
            "House 5-10 Connections"
        ]

        # добавляем колонки для подсчета количества связей домов
        fieldnames += [
            "House 1-10 Count",
            "House 2-5 Count",
            "House 2-10 Count",
            "House 5-10 Count"
        ]
        fieldnames += ["Total House Connections"]

        # добавляем остальные колонки
        fieldnames += [
            "Star Conjunctions",       # список соединений
            "Royal Star Name", # название звезд
            "Bellatrix Star",
            "Other Stars",
            "Check Asc, Sun, Moon"
        ]
   
        with open(results_file, "w", newline="", encoding="utf-8") as out_f:
            writer = csv.DictWriter(out_f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(results)
    
        print(f"\nResults saved to {results_file}")

In [5]:
# Запуск скрипта по таблице вначале вратарей, потом полевых игроков, потом контрольной группы
run_from_table("birth_data.csv")


Processing: Pfaff, Jean Marie

Natal Chart Report for Lebbeke, Belgium
Latitude : 51.0004013
Longitude: 4.1307711
Timezone: Europe/Brussels

Ascendant : 22°21′ Cancer
Midheaven : 24°26′ Pisces

Planets:
Sun     : 12°19′ Sagittarius (House 5)
Moon    : 22°54′ Scorpio (House 5)
Mercury : 22°22′ Scorpio (House 5)
Venus   : 28°51′ Scorpio (House 5)
Mars    : 20°27′ Libra (House 4)
Jupiter : 22°36′ Gemini (House 12)
Saturn  : 4°59′ Scorpio (House 5)
Uranus  : 22°32′ Cancer (House 1)
Neptune : 25°16′ Libra (House 4)
Pluto   : 25°00′ Leo (House 2)

Royal Stars:
Aldebaran : 9°09′ Gemini (House 11)
Regulus   : 29°11′ Leo (House 3)
Antares   : 9°07′ Sagittarius (House 5)
Fomalhaut : 3°12′ Pisces (House 9)
Sirius    : 13°27′ Cancer (House 12)
Algol     : 25°32′ Taurus (House 11)
Rigel     : 16°11′ Gemini (House 11)
Bellatrix : 20°18′ Gemini (House 12)
Capella   : 21°13′ Gemini (House 12)
Betelgeuse: 28°07′ Gemini (House 12)
Canopus   : 14°20′ Cancer (House 12)
Castor    : 19°36′ Cancer (House 12

In [6]:
# Печать содержимого в файл results.csv
import pandas as pd
df = pd.read_csv("results.csv")
print(df)

                     name                                       city  \
0       Pfaff, Jean Marie                           Lebbeke, Belgium   
1       Schmeichel, Peter                        Copenhagen, Denmark   
2            Kahn, Oliver                         Karlsruhe, Germany   
3         Barthez, Fabien                          Lavelanet, France   
4       Buffon, Gianluigi                             Carrara, Italy   
..                    ...                                        ...   
85   Gambineri, Annamaria                                Rome, Italy   
86    Christensen, Haaken                               Oslo, Norway   
87          Reedy, George                      East Chicago, Indiana   
88       Claxton, William  Glendale (Los Angeles County), California   
89  Tallarico, Taj Monroe                    Los Angeles, California   

          date   time          group            Mercury-Mars  \
0   1953-12-04  19:30     goalkeeper                     NaN   
1   196

In [7]:
df.columns

Index(['name', 'city', 'date', 'time', 'group', 'Mercury-Mars',
       'Mercury-Mars Dispositorship/Reception', 'Mercury-Uranus',
       'Mercury-Uranus Dispositorship/Reception', 'Mercury-Saturn',
       'Mercury-Saturn Dispositorship/Reception', 'Mercury-Pluto',
       'Mercury-Pluto Dispositorship/Reception', 'Mars-Uranus',
       'Mars-Uranus Dispositorship/Reception', 'Mars-Pluto',
       'Mars-Pluto Dispositorship/Reception', 'Mars-Moon',
       'Mars-Moon Dispositorship/Reception', 'Mars dignity +/-',
       'Mars 1st House Relation', 'Mars 2-5-10 House Relation',
       'Saturn dignity +', 'Saturn 1st House Relation',
       'Saturn 2-5-10 House Relation', 'Specific Signs Asc/Asc Ruler',
       'Gemini', 'Sagittarius', 'Pisces', 'Aquarius', 'House 1-10 Connections',
       'House 2-5 Connections', 'House 2-10 Connections',
       'House 5-10 Connections', 'House 1-10 Count', 'House 2-5 Count',
       'House 2-10 Count', 'House 5-10 Count', 'Total House Connections',
       'Sta

In [8]:
# --- подсчет статистической значимости для всех параметров
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact

# список колонок для анализа
columns_to_test = [
'Mercury-Mars',
'Mercury-Mars Dispositorship/Reception',
'Mercury-Uranus',
'Mercury-Uranus Dispositorship/Reception',
'Mercury-Saturn',
'Mercury-Saturn Dispositorship/Reception',
'Mercury-Pluto',
'Mercury-Pluto Dispositorship/Reception',
'Mars-Uranus',
'Mars-Uranus Dispositorship/Reception',
'Mars-Pluto',
'Mars-Pluto Dispositorship/Reception',
'Mars-Moon',
'Mars-Moon Dispositorship/Reception',
'Mars dignity +/-',
'Mars 1st House Relation',
'Mars 2-5-10 House Relation',
'Saturn dignity +',
'Saturn 1st House Relation',
'Saturn 2-5-10 House Relation',
'Specific Signs Asc/Asc Ruler',
'Gemini',
'Sagittarius',
'Pisces',
'Aquarius',
'House 1-10 Connections',
'House 2-5 Connections',
'House 2-10 Connections',
'House 5-10 Connections',
'House 1-10 Count',
'House 2-5 Count',
'House 2-10 Count',
'House 5-10 Count',
'Total House Connections',
'Star Conjunctions', 
'Royal Star Name', 
'Bellatrix Star', 
'Other Stars'
]

results = []

# функция превращает любое значение в бинарное (0/1)
def to_binary(x):
    if pd.isna(x):
        return 0
    if isinstance(x, (int, float)):
        return int(x > 0)
    if isinstance(x, str):
        return int(x.strip() != "" and x.lower() != "none")
    return 0

for col in columns_to_test:

    temp = df[['group', col]].copy()

    # создаем бинарную колонку
    temp['value'] = temp[col].apply(to_binary)

    contingency = pd.crosstab(temp['group'], temp['value'])

    # если нет двух колонок (0 и 1) → пропускаем
    if contingency.shape[1] < 2:
        continue

    # χ² для всех групп
    chi2_all, p_chi2_all, _, _ = chi2_contingency(contingency)

    # GK vs Field
    if 'goalkeeper' in contingency.index and 'outfield player' in contingency.index:
        pair = contingency.loc[['goalkeeper','outfield player']]
        chi2_gk_field, p_chi2_gk_field, _, _ = chi2_contingency(pair)
        _, p_fisher_gk_field = fisher_exact(pair.values)
    else:
        p_chi2_gk_field = None
        p_fisher_gk_field = None

    # GK vs Control
    if 'goalkeeper' in contingency.index and 'control group' in contingency.index:
        pair = contingency.loc[['goalkeeper','control group']]
        chi2_gk_control, p_chi2_gk_control, _, _ = chi2_contingency(pair)
        _, p_fisher_gk_control = fisher_exact(pair.values)
    else:
        p_chi2_gk_control = None
        p_fisher_gk_control = None

    # проценты
    percents = contingency.div(temp.groupby('group').size(), axis=0) * 100

    results.append({
        'variable': col,
        'χ² p-value (all groups)': round(p_chi2_all,3),
        'χ² p-value GK vs Field': round(p_chi2_gk_field,3) if p_chi2_gk_field else None,
        'χ² p-value GK vs Control': round(p_chi2_gk_control,3) if p_chi2_gk_control else None,
        'Fisher p-value GK vs Field': round(p_fisher_gk_field,3) if p_fisher_gk_field else None,
        'Fisher p-value GK vs Control': round(p_fisher_gk_control,3) if p_fisher_gk_control else None,
        'GK %': round(percents.loc['goalkeeper',1],1) if 'goalkeeper' in percents.index else None,
        'Field %': round(percents.loc['outfield player',1],1) if 'outfield player' in percents.index else None,
        'Control %': round(percents.loc['control group',1],1) if 'control group' in percents.index else None
    })

results_df = pd.DataFrame(results)

# сортируем по общей значимости
results_df = results_df.sort_values('χ² p-value (all groups)')

print(results_df)

# --- генерация текстовых инсайтов
# Берёт таблицу results_df
# Выбирает только статистически значимые результаты
# Сразу печатает готовые формулировки для исследования.

def generate_insight(row):

    var = row['variable']
    gk = row['GK %']
    field = row['Field %']
    control = row['Control %']

    fisher_field = row['Fisher p-value GK vs Field']
    fisher_control = row['Fisher p-value GK vs Control']

    text = f"{var} чаще встречается у вратарей ({gk}%), "
    text += f"у полевых игроков {field}%, "
    text += f"в контрольной группе {control}%. "

    # GK vs Field
    if pd.notna(fisher_field):
        prob = round(fisher_field * 100,1)
        if fisher_field < 0.05:
            text += f"Разница между вратарями и полевыми статистически значима (вероятность случайного совпадения {prob}%). "
        else:
            text += f"Разница между вратарями и полевыми статистически незначима ({prob}% вероятность случайного возникновения). "

    # GK vs Control
    if pd.notna(fisher_control):
        prob = round(fisher_control * 100,1)
        if fisher_control < 0.05:
            text += f"Сравнение с контрольной группой показало статистически значимое отличие (вероятность случайного совпадения {prob}%)."
        else:
            text += f"Разница с контрольной группой статистически незначима ({prob}% вероятность случайного возникновения)."

    return text


# --- фильтр значимых результатов
significant = results_df[
    (results_df['Fisher p-value GK vs Field'] < 0.05) |
    (results_df['Fisher p-value GK vs Control'] < 0.05) |
    (results_df['χ² p-value (all groups)'] < 0.05)
]

# --- печать инсайтов
for _, row in significant.iterrows():
    print(generate_insight(row))
    print()

                                   variable  χ² p-value (all groups)  \
9      Mars-Uranus Dispositorship/Reception                    0.019   
36                           Bellatrix Star                    0.036   
29                         House 1-10 Count                    0.054   
25                   House 1-10 Connections                    0.054   
5   Mercury-Saturn Dispositorship/Reception                    0.093   
14                         Mars dignity +/-                    0.097   
37                              Other Stars                    0.149   
34                        Star Conjunctions                    0.154   
32                         House 5-10 Count                    0.179   
28                   House 5-10 Connections                    0.179   
4                            Mercury-Saturn                    0.186   
19             Saturn 2-5-10 House Relation                    0.259   
16               Mars 2-5-10 House Relation                    0

In [9]:
# сохраняем статистическую проверку каждого параметра в файл
significant.to_csv("statistic_check_single_parameter.csv", index=False)

In [10]:
"""
Что делает этот код:
Берёт все комбинации аспектов (2–7) и считает наличие связки.
Проверяет Fisher exact test для:
Вратари vs Полевые
Вратари vs Контрольная группа
Сохраняет:
Связки значимые только для GK vs Field
Связки значимые только для GK vs Control
Связки значимые для обоих сравнений одновременно
Безопасно выводит, если значимых связок нет.
"""

import pandas as pd
from itertools import combinations
from scipy.stats import fisher_exact

# колонки аспектов
aspect_columns = [
    'Mercury-Mars',
    'Mercury-Uranus',
    'Mercury-Saturn',
    'Mercury-Pluto',
    'Mars-Uranus',
    'Mars-Pluto',
    'Mars-Moon'
]

# бинаризация: текст = 1, NaN = 0
def to_binary(x):
    return 0 if pd.isna(x) else 1

binary_df = df[['group'] + aspect_columns].copy()
for col in aspect_columns:
    binary_df[col] = binary_df[col].apply(to_binary)

# списки для результатов
significant_gk_field = []
significant_gk_control = []
significant_both = []

# генерация комбинаций от 2 до 7 аспектов
for r in range(2, len(aspect_columns)+1):
    for combo in combinations(aspect_columns, r):
        combo_name = " & ".join(combo)
        binary_df['combo'] = binary_df[list(combo)].all(axis=1).astype(int)
        
        # Fisher GK vs Field
        if 'goalkeeper' in binary_df['group'].values and 'outfield player' in binary_df['group'].values:
            pair_field = pd.crosstab(binary_df['group'], binary_df['combo']).loc[['goalkeeper','outfield player']]
            p_fisher_field = 1.0 if pair_field.shape[1]==1 else fisher_exact(pair_field.values)[1]
        else:
            p_fisher_field = None
        
        # Fisher GK vs Control
        if 'goalkeeper' in binary_df['group'].values and 'control group' in binary_df['group'].values:
            pair_control = pd.crosstab(binary_df['group'], binary_df['combo']).loc[['goalkeeper','control group']]
            p_fisher_control = 1.0 if pair_control.shape[1]==1 else fisher_exact(pair_control.values)[1]
        else:
            p_fisher_control = None
        
        percents = binary_df.groupby('group')['combo'].mean()*100
        
        # собираем значимые по сравнению с полевыми
        if p_fisher_field is not None and p_fisher_field < 0.05:
            significant_gk_field.append({
                'link': combo_name,
                'Fisher_p_GK_vs_Field': round(p_fisher_field,3),
                'GK_%': round(percents.get('goalkeeper',0),1),
                'Field_%': round(percents.get('outfield player',0),1)
            })
        
        # собираем значимые по сравнению с контрольной
        if p_fisher_control is not None and p_fisher_control < 0.05:
            significant_gk_control.append({
                'link': combo_name,
                'Fisher_p_GK_vs_Control': round(p_fisher_control,3),
                'GK_%': round(percents.get('goalkeeper',0),1),
                'Control_%': round(percents.get('control group',0),1)
            })
        
        # значимые в обоих сравнениях
        if p_fisher_field is not None and p_fisher_control is not None:
            if p_fisher_field < 0.05 and p_fisher_control < 0.05:
                significant_both.append({
                    'link': combo_name,
                    'Fisher_p_GK_vs_Field': round(p_fisher_field,3),
                    'Fisher_p_GK_vs_Control': round(p_fisher_control,3),
                    'GK_%': round(percents.get('goalkeeper',0),1),
                    'Field_%': round(percents.get('outfield player',0),1),
                    'Control_%': round(percents.get('control group',0),1)
                })

# --- вывод результатов
print("=== Значимые связки GK vs Field ===")
if significant_gk_field:
    df_field = pd.DataFrame(significant_gk_field).sort_values('Fisher_p_GK_vs_Field')
    print(df_field)
else:
    print("Нет значимых связок для GK vs Field.")

print("\n=== Значимые связки GK vs Control ===")
if significant_gk_control:
    df_control = pd.DataFrame(significant_gk_control).sort_values('Fisher_p_GK_vs_Control')
    print(df_control)
else:
    print("Нет значимых связок для GK vs Control.")

print("\n=== Значимые связки для обоих сравнений одновременно ===")
if significant_both:
    df_both = pd.DataFrame(significant_both).sort_values('Fisher_p_GK_vs_Control')
    print(df_both)
else:
    print("Нет связок, которые значимы одновременно для GK vs Field и GK vs Control.")

=== Значимые связки GK vs Field ===
Нет значимых связок для GK vs Field.

=== Значимые связки GK vs Control ===
Нет значимых связок для GK vs Control.

=== Значимые связки для обоих сравнений одновременно ===
Нет связок, которые значимы одновременно для GK vs Field и GK vs Control.


In [11]:
"""
Оптимизируем код под маленькую выборку:
Комбинации аспектов от 2 до 3, чтобы увеличить шансы на встречаемость.
Фильтруем редкие комбинации: сохраняем только комбинации, где хотя бы 3 карты у вратарей имеют эту связку.
"""

import pandas as pd
from itertools import combinations
from scipy.stats import fisher_exact

# колонки аспектов
aspect_columns = [
    'Mercury-Mars',
    'Mercury-Uranus',
    'Mercury-Saturn',
    'Mercury-Pluto',
    'Mars-Uranus',
    'Mars-Pluto',
    'Mars-Moon'
]

def to_binary(x):
    return 0 if pd.isna(x) else 1

# бинаризация
binary_df = df[['group'] + aspect_columns].copy()
for col in aspect_columns:
    binary_df[col] = binary_df[col].apply(to_binary)

# результаты
significant_gk_field = []
significant_gk_control = []
significant_both = []

# комбинации 2–3 аспектов
for r in range(2, 4):
    for combo in combinations(aspect_columns, r):
        combo_name = " & ".join(combo)
        # 1, если все аспекты в связке есть
        binary_df['combo'] = binary_df[list(combo)].all(axis=1).astype(int)
        
        # фильтр: хотя бы 3 карты у вратарей
        if binary_df[binary_df['group']=='goalkeeper']['combo'].sum() < 3:
            continue
        
        # Fisher GK vs Field
        if 'goalkeeper' in binary_df['group'].values and 'outfield player' in binary_df['group'].values:
            pair_field = pd.crosstab(binary_df['group'], binary_df['combo']).loc[['goalkeeper','outfield player']]
            p_fisher_field = 1.0 if pair_field.shape[1]==1 else fisher_exact(pair_field.values)[1]
        else:
            p_fisher_field = None
        
        # Fisher GK vs Control
        if 'goalkeeper' in binary_df['group'].values and 'control group' in binary_df['group'].values:
            pair_control = pd.crosstab(binary_df['group'], binary_df['combo']).loc[['goalkeeper','control group']]
            p_fisher_control = 1.0 if pair_control.shape[1]==1 else fisher_exact(pair_control.values)[1]
        else:
            p_fisher_control = None
        
        percents = binary_df.groupby('group')['combo'].mean()*100
        
        # значимые для GK vs Field
        if p_fisher_field is not None and p_fisher_field < 0.05:
            significant_gk_field.append({
                'link': combo_name,
                'Fisher_p_GK_vs_Field': round(p_fisher_field,3),
                'GK_%': round(percents.get('goalkeeper',0),1),
                'Field_%': round(percents.get('outfield player',0),1)
            })
        
        # значимые для GK vs Control
        if p_fisher_control is not None and p_fisher_control < 0.05:
            significant_gk_control.append({
                'link': combo_name,
                'Fisher_p_GK_vs_Control': round(p_fisher_control,3),
                'GK_%': round(percents.get('goalkeeper',0),1),
                'Control_%': round(percents.get('control group',0),1)
            })
        
        # значимые для обоих
        if p_fisher_field is not None and p_fisher_control is not None:
            if p_fisher_field < 0.05 and p_fisher_control < 0.05:
                significant_both.append({
                    'link': combo_name,
                    'Fisher_p_GK_vs_Field': round(p_fisher_field,3),
                    'Fisher_p_GK_vs_Control': round(p_fisher_control,3),
                    'GK_%': round(percents.get('goalkeeper',0),1),
                    'Field_%': round(percents.get('outfield player',0),1),
                    'Control_%': round(percents.get('control group',0),1)
                })

# --- вывод
print("=== Значимые связки GK vs Field ===")
if significant_gk_field:
    df_field = pd.DataFrame(significant_gk_field).sort_values('Fisher_p_GK_vs_Field')
    print(df_field)
else:
    print("Нет значимых связок для GK vs Field.")

print("\n=== Значимые связки GK vs Control ===")
if significant_gk_control:
    df_control = pd.DataFrame(significant_gk_control).sort_values('Fisher_p_GK_vs_Control')
    print(df_control)
else:
    print("Нет значимых связок для GK vs Control.")

print("\n=== Значимые связки для обоих сравнений одновременно ===")
if significant_both:
    df_both = pd.DataFrame(significant_both).sort_values('Fisher_p_GK_vs_Control')
    print(df_both)
else:
    print("Нет связок, которые значимы одновременно для GK vs Field и GK vs Control.")

=== Значимые связки GK vs Field ===
Нет значимых связок для GK vs Field.

=== Значимые связки GK vs Control ===
Нет значимых связок для GK vs Control.

=== Значимые связки для обоих сравнений одновременно ===
Нет связок, которые значимы одновременно для GK vs Field и GK vs Control.


In [12]:
# Подсчет статистической значимости наличия звезды Bellatrix Star
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact

# --- выбираем только нужные колонки и убираем пустые значения
stars_df = (
    df[['group','Bellatrix Star']]
    .dropna()
    .assign(star=lambda x: x['Bellatrix Star'].str.split(','))
    .explode('star')
)

stars_df['star'] = stars_df['star'].str.strip()
unique_stars = stars_df['star'].unique()
significant_stars = []

for star in unique_stars:

    # --- функция для безопасной проверки наличия звезды
    def has_star(cell, star_name):
        if pd.isna(cell):
            return 0
        if isinstance(cell, str):
            return int(star_name in [s.strip() for s in cell.split(',')])
        elif isinstance(cell, list):
            return int(star_name in [s.strip() for s in cell])
        return 0

    df['has_star'] = df['Bellatrix Star'].apply(lambda x: has_star(x, star))
    
    contingency = pd.crosstab(df['group'], df['has_star'])

    # --- χ² тест для всех групп
    chi2_all, p_chi2_all, _, _ = chi2_contingency(contingency)

    # --- χ² тест: Вратари vs Полевые
    if 'goalkeeper' in contingency.index and 'outfield player' in contingency.index:
        chi2_gk_field, p_chi2_gk_field, _, _ = chi2_contingency(contingency.loc[['goalkeeper','outfield player']])
        table_gk_field = contingency.loc[['goalkeeper','outfield player']].values
        _, p_fisher_gk_field = fisher_exact(table_gk_field)
    else:
        p_chi2_gk_field = p_fisher_gk_field = None

    # --- χ² тест: Вратари vs Контрольная группа
    if 'goalkeeper' in contingency.index and 'control group' in contingency.index:
        chi2_gk_control, p_chi2_gk_control, _, _ = chi2_contingency(contingency.loc[['goalkeeper','control group']])
        table_gk_control = contingency.loc[['goalkeeper','control group']].values
        _, p_fisher_gk_control = fisher_exact(table_gk_control)
    else:
        p_chi2_gk_control = p_fisher_gk_control = None

    percents = contingency.div(df.groupby('group').size(), axis=0)[1]*100 if 1 in contingency.columns else pd.Series()

    significant_stars.append({
        'star': star,
        'χ² p-value (all groups)': round(p_chi2_all,3),
        'χ² p-value GK vs Field': round(p_chi2_gk_field,3) if p_chi2_gk_field is not None else None,
        'χ² p-value GK vs Control': round(p_chi2_gk_control,3) if p_chi2_gk_control is not None else None,
        'Fisher p-value GK vs Field': round(p_fisher_gk_field,3) if p_fisher_gk_field is not None else None,
        'Fisher p-value GK vs Control': round(p_fisher_gk_control,3) if p_fisher_gk_control is not None else None,
        'goalkeeper count': contingency.loc['goalkeeper',1] if 1 in contingency.columns and 'goalkeeper' in contingency.index else 0,
        'goalkeeper %': round(percents.get('goalkeeper',0),1),
        'field player count': contingency.loc['outfield player',1] if 1 in contingency.columns and 'outfield player' in contingency.index else 0,
        'field player %': round(percents.get('outfield player',0),1),
        'control count': contingency.loc['control group',1] if 1 in contingency.columns and 'control group' in contingency.index else 0,
        'control %': round(percents.get('control group',0),1)
    })

# --- итоговая таблица
significant_df = pd.DataFrame(significant_stars)
significant_df = significant_df.sort_values('χ² p-value (all groups)')

print(significant_df)

        star  χ² p-value (all groups)  χ² p-value GK vs Field  \
0  Bellatrix                    0.036                    0.47   

   χ² p-value GK vs Control  Fisher p-value GK vs Field  \
0                     0.031                       0.472   

   Fisher p-value GK vs Control  goalkeeper count  goalkeeper %  \
0                         0.024                 6          20.0   

   field player count  field player %  control count  control %  
0                   3            10.0              0        0.0  


In [13]:
# Проверка статистической значимости для комбиначий аспектов
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# 1. Считываем данные
df = pd.read_csv('results.csv', encoding='utf-8-sig')

# Список аспектов для анализа
columns_to_check = [
    'Mercury-Mars', 'Mercury-Uranus', 'Mercury-Saturn', 'Mercury-Pluto',
    'Mars-Uranus', 'Mars-Pluto', 'Mars-Moon'
]

# 2. Подготовка маски (наличие текста = аспект есть)
def has_aspect(val):
    s = str(val).lower().strip()
    return len(s) > 2 and s not in ['nan', 'none', '0', '0.0', '1', '1.0']

# Группы
gk_mask = df['group'] == 'goalkeeper'
ctrl_mask = df['group'] == 'control group'

n_gk = gk_mask.sum()
n_ctrl = ctrl_mask.sum()

stats_results = []

from itertools import combinations

# --- 3. Расчет теста Фишера для КОМБИНАЦИЙ ---
results = []

# Перебираем комбинации от 2 до 7 аспектов
for r in range(2, len(columns_to_check) + 1):
    for combo in combinations(columns_to_check, r):
        # Проверяем наличие ВСЕХ аспектов из комбинации одновременно
        # (используем вашу функцию has_aspect)
        aspects_present = df[list(combo)].map(has_aspect).all(axis=1)
        
        gk_with = (aspects_present & gk_mask).sum()
        gk_without = n_gk - gk_with
        
        ctrl_with = (aspects_present & ctrl_mask).sum()
        ctrl_without = n_ctrl - ctrl_with
        
        # Чтобы не считать пустые строки, где у вратарей 0 таких комбинаций
        if gk_with > 0:
            table = [[gk_with, gk_without], [ctrl_with, ctrl_without]]
            odds_ratio, p_val = fisher_exact(table)
            
            results.append({
                'Combination': " + ".join(combo),
                'GK_count': gk_with,
                'Ctrl_count': ctrl_with,
                'p_value': p_val,
                'Odds_Ratio': odds_ratio
            })

# --- 4. Анализ значимости ---
stats_df = pd.DataFrame(results)

# Фильтруем значимые (p < 0.05)
significant = stats_df[stats_df['p_value'] < 0.05].sort_values('p_value')

if not significant.empty:
    print("=== Статистически значимые КОМБИНАЦИИ (p < 0.05) ===")
    print(significant.to_string(index=False))
else:
    print("Значимых комбинаций не обнаружено.")
    if not stats_df.empty:
        print(f"Минимальный p-value среди комбинаций: {stats_df['p_value'].min():.4f}")

# Поправка на множественные сравнения (Бонферрони или Холм)
# Если аспектов мало, можно смотреть на сырой p-value, но для науки лучше скорректировать
stats_df['p_significant'] = stats_df['p_value'] < 0.05

# Вывод только значимых
significant_aspects = stats_df[stats_df['p_significant'] == True].sort_values('p_value')

if not significant_aspects.empty:
    print("=== Статистически значимые аспекты (p < 0.05) ===")
    print(significant_aspects[['Aspect', 'GK_count', 'Ctrl_count', 'p_value', 'Odds_Ratio']])
else:
    print("Значимых различий при p < 0.05 не обнаружено.")
    print("\nМинимальный найденный p-value:", stats_df['p_value'].min())

Значимых комбинаций не обнаружено.
Минимальный p-value среди комбинаций: 0.4238
Значимых различий при p < 0.05 не обнаружено.

Минимальный найденный p-value: 0.42380990998296975


In [14]:
# Поиск частоты наиболее часто встречающихся комбинаций
import pandas as pd
from itertools import combinations
import re

# --- 1. ЧИТАЕМ ФАЙЛ ---
try:
    # utf-8-sig помогает, если в файле есть символы градусов (°)
    df = pd.read_csv('results.csv', encoding='utf-8-sig')
    print("Файл прочитан. Строк:", len(df))
except FileNotFoundError:
    print("Ошибка: Файл 'results.csv' не найден в папке с кодом.")
    exit()

# --- 2. ПОИСК НУЖНЫХ КОЛОНОК ---
# Список аспектов, которые мы ищем
aspect_pairs = [
    'Mercury-Mars', 'Mercury-Uranus', 'Mercury-Saturn', 'Mercury-Pluto',
    'Mars-Uranus', 'Mars-Pluto', 'Mars-Moon'
]
# Берем только те колонки, которые реально есть в CSV
cols = [c for c in aspect_pairs if c in df.columns]

if not cols:
    print("В файле не найдены колонки из списка. Доступные колонки:", df.columns.tolist())
    exit()

# --- 3. ПОДГОТОВКА МАСОК ---
def is_real_aspect(val):
    """Считает ячейку заполненной, только если там есть текст названия аспекта."""
    s = str(val).lower().strip()
    # Игнорируем: '1', '0', 'nan', 'none' и пустые строки
    if s in ['1', '0', 'nan', 'none', '', '0.0', '1.0']:
        return False
    # Ищем наличие букв (Trine, Square, etc.)
    return any(c.isalpha() for c in s)

# Создаем карту 'True/False' для всех ячеек
aspect_mask_df = df[cols].map(is_real_aspect)

# Маски групп
is_gk = df['group'] == 'goalkeeper'
is_out = df['group'] == 'outfield player'
is_ctrl = df['group'] == 'control group'

# Общее количество для расчета %
counts = {
    'GK': is_gk.sum(),
    'Out': is_out.sum(),
    'Ctrl': is_ctrl.sum()
}

# --- 4. ПОИСК КОМБИНАЦИЙ ---
results = []
for r in range(2, len(cols) + 1):
    for combo in combinations(cols, r):
        # Строка подходит, если во ВСЕХ колонках набора найден текст аспекта
        has_block = aspect_mask_df[list(combo)].all(axis=1)
        
        gk_n = int(has_block[is_gk].sum())
        
        # Нас интересуют только те блоки, которые есть у вратарей
        if gk_n > 0:
            results.append({
                'block': " + ".join(combo),
                'GK n': gk_n,
                'GK %': round(gk_n / counts['GK'] * 100, 1) if counts['GK'] > 0 else 0,
                'Out %': round(has_block[is_out].sum() / counts['Out'] * 100, 1) if counts['Out'] > 0 else 0,
                'Ctrl %': round(has_block[is_ctrl].sum() / counts['Ctrl'] * 100, 1) if counts['Ctrl'] > 0 else 0
            })

# --- 5. ВЫВОД ---
res_df = pd.DataFrame(results)
if not res_df.empty:
    # Сортировка по убыванию частоты у вратарей
    print(res_df.sort_values('GK n', ascending=False).head(30).to_string(index=False))
    # Сохраним в новый файл, чтобы вы могли его открыть в Excel
    res_df.to_csv('frequency_few_aspects_check.csv', index=False, encoding='utf-8-sig')
    print("\nГотово! Результат сохранен в 'frequency_few_aspects_check.csv'")
else:
    print("Комбинаций текстовых аспектов не найдено.")

Файл прочитан. Строк: 90
                                                     block  GK n  GK %  Out %  Ctrl %
                                 Mercury-Pluto + Mars-Moon     5  16.7   16.7    13.3
                            Mercury-Uranus + Mercury-Pluto     5  16.7   13.3     6.7
                             Mercury-Mars + Mercury-Uranus     3  10.0    6.7     6.7
                              Mercury-Uranus + Mars-Uranus     3  10.0    6.7     3.3
                              Mercury-Mars + Mercury-Pluto     3  10.0   10.0    10.0
                                    Mars-Pluto + Mars-Moon     3  10.0   16.7    10.0
                                   Mars-Uranus + Mars-Moon     3  10.0   13.3    10.0
                                Mercury-Pluto + Mars-Pluto     3  10.0   10.0    10.0
                                  Mercury-Mars + Mars-Moon     3  10.0   13.3     6.7
                                Mercury-Mars + Mars-Uranus     3  10.0    6.7     6.7
                             

In [15]:
# --- 5. ВЫВОД И ФИЛЬТРАЦИЯ  только тех карт, которые преобладают у вратарей ---
res_df = pd.DataFrame(results)

if not res_df.empty:
    # 1. Вывод топ-30 (как и было)
    print("\n=== Топ-30 комбинаций по количеству у вратарей ===")
    print(res_df.sort_values('GK n', ascending=False).head(30).to_string(index=False))
    
    # 2. НОВЫЙ БЛОК: Вывод комбинаций, где у вратарей % выше, чем у остальных
    print("\n=== Комбинации, преобладающие у вратарей (GK % > Out % И GK % > Ctrl %) ===")
    
    # Создаем фильтр: процент у GK должен быть строго больше, чем у полевых И контроля
    predominant_gk = res_df[(res_df['GK %'] > res_df['Out %']) & (res_df['GK %'] > res_df['Ctrl %'])]
    
    if not predominant_gk.empty:
        # Сортируем по силе различия (разница между GK и ближайшим конкурентом)
        predominant_gk = predominant_gk.sort_values('GK %', ascending=False)
        print(predominant_gk.to_string(index=False))
    else:
        print("Комбинаций с преобладанием у вратарей не обнаружено.")

    res_df.to_csv('frequency_few_aspects_check.csv', index=False, encoding='utf-8-sig')
    print("\nГотово! Полный результат сохранен в 'frequency_few_aspects_check.csv'")
else:
    print("Комбинаций текстовых аспектов не найдено.")


=== Топ-30 комбинаций по количеству у вратарей ===
                                                     block  GK n  GK %  Out %  Ctrl %
                                 Mercury-Pluto + Mars-Moon     5  16.7   16.7    13.3
                            Mercury-Uranus + Mercury-Pluto     5  16.7   13.3     6.7
                             Mercury-Mars + Mercury-Uranus     3  10.0    6.7     6.7
                              Mercury-Uranus + Mars-Uranus     3  10.0    6.7     3.3
                              Mercury-Mars + Mercury-Pluto     3  10.0   10.0    10.0
                                    Mars-Pluto + Mars-Moon     3  10.0   16.7    10.0
                                   Mars-Uranus + Mars-Moon     3  10.0   13.3    10.0
                                Mercury-Pluto + Mars-Pluto     3  10.0   10.0    10.0
                                  Mercury-Mars + Mars-Moon     3  10.0   13.3     6.7
                                Mercury-Mars + Mars-Uranus     3  10.0    6.7     6.7
  

In [16]:
# поиск статистически значимых комбинаций для вратарей - по сравнению с полевыми игроками
# добавила диспозиторы и специфические знаки на Асц в список
import pandas as pd
from scipy.stats import fisher_exact
from itertools import combinations

# 1. Считываем данные
df = pd.read_csv('results.csv', encoding='utf-8-sig')

columns_to_check = [
    'Mercury-Mars', 'Mercury-Uranus', 'Mercury-Saturn', 
    'Mercury-Pluto', 'Mars-Uranus', 'Mars-Pluto', 'Mars-Moon',
    'Mars dignity +/-', 'Mars 1st House Relation', 'Mars 2-5-10 House Relation',
    'Saturn dignity +', 'Saturn 1st House Relation',
    'Specific Signs Asc/Asc Ruler',
    'Gemini', 'Sagittarius', 'Pisces', 'Aquarius', 'House 1-10 Connections',
    'House 2-5 Connections', 'House 2-10 Connections',
    'House 5-10 Connections', 'Royal Star Name', 'Bellatrix Star'
]

# 2. Подготовка данных
def has_aspect_func(val):
    s = str(val).lower().strip()
    return len(s) > 2 and s not in ['nan', 'none', '0', '0.0', '1', '1.0']

# Предварительный расчет булевой матрицы (ускоряет работу в разы)
df_bool = df[columns_to_check].map(has_aspect_func)

gk_mask = df['group'] == 'goalkeeper'
outfield_mask = df['group'] == 'outfield player'

n_gk = gk_mask.sum()
n_ctrl = outfield_mask.sum()

# --- 3. Расчет теста Фишера ---
results = []
total_processed = 0

for r in range(2, 7): 
    print(f"\nАнализ комбинаций по {r} элемента(ов)...")
    for combo in combinations(columns_to_check, r):
        aspects_present = df_bool[list(combo)].all(axis=1)
        gk_with = (aspects_present & gk_mask).sum()
        
        # Считаем только те комбинации, которые есть хотя бы у одного вратаря
        if gk_with > 0:
            gk_without = n_gk - gk_with
            ctrl_with = (aspects_present & outfield_mask).sum()
            ctrl_without = n_ctrl - ctrl_with
            
            table = [[gk_with, gk_without], [ctrl_with, ctrl_without]]
            odds_ratio, p_val = fisher_exact(table)
            
            results.append({
                'Combination': " + ".join(combo),
                'GK_count': gk_with,
                'Outfield_count': ctrl_with,
                'p_value': p_val,
                'Odds_Ratio': odds_ratio
            })
        
        total_processed += 1
        if total_processed % 50000 == 0:
            print(f"Проверено {total_processed} комбинаций...")

# --- 4. Анализ и сохранение результатов ---
stats_df = pd.DataFrame(results)

if not stats_df.empty:
    # Сортируем по p-value (самые значимые наверху)
    stats_df = stats_df.sort_values('p_value')
    
    # Выбираем значимые (p < 0.05)
    significant = stats_df[stats_df['p_value'] < 0.05]
    
    # 1. ПЕЧАТЬ В КОНСОЛЬ
    if not significant.empty:
        print("\n" + "="*70)
        print(f"НАЙДЕНО {len(significant)} СТАТИСТИЧЕСКИ ЗНАЧИМЫХ КОМБИНАЦИЙ (p < 0.05)")
        print("="*70)
        print(significant[['Combination', 'GK_count', 'Outfield_count', 'p_value', 'Odds_Ratio']].to_string(index=False))
    else:
        print("\nЗначимых комбинаций (p < 0.05) не обнаружено.")
        print(f"Минимальный p-value: {stats_df['p_value'].min():.4f}")
        print("Топ-10 наиболее близких комбинаций выведены в файл.")

    # 2. СОХРАНЕНИЕ В ФАЙЛ
    # Если значимых много - сохраняем все значимые. 
    # Если значимых нет - сохраняем топ-100 "почти значимых".
    to_save = significant if not significant.empty else stats_df.head(100)
    
    filename = 'goalkeeper_outfield_analysis_results.csv'
    to_save.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"\nДанные сохранены в файл: {filename}")

else:
    print("\nНет данных для анализа (список результатов пуст).")


Анализ комбинаций по 2 элемента(ов)...

Анализ комбинаций по 3 элемента(ов)...

Анализ комбинаций по 4 элемента(ов)...

Анализ комбинаций по 5 элемента(ов)...

Анализ комбинаций по 6 элемента(ов)...
Проверено 50000 комбинаций...
Проверено 100000 комбинаций...

НАЙДЕНО 17 СТАТИСТИЧЕСКИ ЗНАЧИМЫХ КОМБИНАЦИЙ (p < 0.05)
                                                                                                                        Combination  GK_count  Outfield_count  p_value  Odds_Ratio
                        Mars 2-5-10 House Relation + Specific Signs Asc/Asc Ruler + House 1-10 Connections + House 5-10 Connections        12               2 0.004769    9.333333
                                                     Specific Signs Asc/Asc Ruler + House 1-10 Connections + House 5-10 Connections        16               5 0.006107    5.714286
                                                                                    House 1-10 Connections + House 5-10 Connections        17    

In [17]:
# поиск статистически значимых комбинаций для вратарей - контрольная группа
# добавила диспозиторы и специфические знаки на Асц в список
import pandas as pd
from scipy.stats import fisher_exact
from itertools import combinations

# 1. Считываем данные
df = pd.read_csv('results.csv', encoding='utf-8-sig')

columns_to_check = [
    'Mercury-Mars', 'Mercury-Uranus', 'Mercury-Saturn', 
    'Mercury-Pluto', 'Mars-Uranus', 'Mars-Pluto', 'Mars-Moon',
    'Mars dignity +/-', 'Mars 1st House Relation', 'Mars 2-5-10 House Relation',
    'Saturn dignity +', 'Saturn 1st House Relation',
    'Specific Signs Asc/Asc Ruler',
    'Gemini', 'Sagittarius', 'Pisces', 'Aquarius', 'House 1-10 Connections',
    'House 2-5 Connections', 'House 2-10 Connections',
    'House 5-10 Connections', 'Royal Star Name', 'Bellatrix Star'
]

# 2. Подготовка данных
def has_aspect_func(val):
    s = str(val).lower().strip()
    return len(s) > 2 and s not in ['nan', 'none', '0', '0.0', '1', '1.0']

# Предварительный расчет булевой матрицы (ускоряет работу в разы)
df_bool = df[columns_to_check].map(has_aspect_func)

gk_mask = df['group'] == 'goalkeeper'
ctrl_mask = df['group'] == 'control group'

n_gk = gk_mask.sum()
n_ctrl = ctrl_mask.sum()

# --- 3. Расчет теста Фишера ---
results = []
total_processed = 0

for r in range(2, 7): 
    print(f"\nАнализ комбинаций по {r} элемента(ов)...")
    for combo in combinations(columns_to_check, r):
        aspects_present = df_bool[list(combo)].all(axis=1)
        gk_with = (aspects_present & gk_mask).sum()
        
        # Считаем только те комбинации, которые есть хотя бы у одного вратаря
        if gk_with > 0:
            gk_without = n_gk - gk_with
            ctrl_with = (aspects_present & ctrl_mask).sum()
            ctrl_without = n_ctrl - ctrl_with
            
            table = [[gk_with, gk_without], [ctrl_with, ctrl_without]]
            odds_ratio, p_val = fisher_exact(table)
            
            results.append({
                'Combination': " + ".join(combo),
                'GK_count': gk_with,
                'Ctrl_count': ctrl_with,
                'p_value': p_val,
                'Odds_Ratio': odds_ratio
            })
        
        total_processed += 1
        if total_processed % 50000 == 0:
            print(f"Проверено {total_processed} комбинаций...")

# --- 4. Анализ значимости ---
stats_df = pd.DataFrame(results)

if not stats_df.empty:
    significant = stats_df[stats_df['p_value'] < 0.05].sort_values('p_value')

    if not significant.empty:
        print("\n" + "="*50)
        print("СТАТИСТИЧЕСКИ ЗНАЧИМЫЕ КОМБИНАЦИИ (p < 0.05)")
        print("="*50)
        print(significant[['Combination', 'GK_count', 'Ctrl_count', 'p_value', 'Odds_Ratio']].to_string(index=False))
    else:
        print("\nЗначимых комбинаций не обнаружено.")
        print(f"Минимальный p-value: {stats_df['p_value'].min():.4f}")
else:
    print("\nНет данных для анализа.")


Анализ комбинаций по 2 элемента(ов)...

Анализ комбинаций по 3 элемента(ов)...

Анализ комбинаций по 4 элемента(ов)...

Анализ комбинаций по 5 элемента(ов)...

Анализ комбинаций по 6 элемента(ов)...
Проверено 50000 комбинаций...
Проверено 100000 комбинаций...

СТАТИСТИЧЕСКИ ЗНАЧИМЫЕ КОМБИНАЦИИ (p < 0.05)
                                                                                      Combination  GK_count  Ctrl_count  p_value  Odds_Ratio
                                                    Specific Signs Asc/Asc Ruler + Bellatrix Star         6           0 0.023721         inf
                               Mercury-Mars + Mars 2-5-10 House Relation + House 1-10 Connections         6           0 0.023721         inf
Mercury-Mars + Mars 2-5-10 House Relation + Specific Signs Asc/Asc Ruler + House 1-10 Connections         6           0 0.023721         inf
                                                               Mars-Pluto + House 2-5 Connections         2           9 0.041900 

In [18]:
# поиск статистически значимых комбинаций как для вратарей - контрольная группа
# так и для вратарей - полевые игроки
# добавила диспозиторы и специфические знаки на Асц в список
import pandas as pd
from scipy.stats import fisher_exact
from itertools import combinations

# 1. Считываем данные
df = pd.read_csv('results.csv', encoding='utf-8-sig')

columns_to_check = [
    'Mercury-Mars',
       'Mercury-Mars Dispositorship/Reception', 'Mercury-Uranus',
       'Mercury-Uranus Dispositorship/Reception', 'Mercury-Saturn',
       'Mercury-Saturn Dispositorship/Reception', 'Mercury-Pluto',
       'Mercury-Pluto Dispositorship/Reception', 'Mars-Uranus',
       'Mars-Uranus Dispositorship/Reception', 'Mars-Pluto',
       'Mars-Pluto Dispositorship/Reception', 'Mars-Moon',
       'Mars-Moon Dispositorship/Reception',
    'Mars dignity +/-', 'Mars 1st House Relation', 'Mars 2-5-10 House Relation',
    'Saturn dignity +', 'Saturn 1st House Relation',
    'Specific Signs Asc/Asc Ruler',
    'Gemini', 'Sagittarius', 'Pisces', 'Aquarius', 'House 1-10 Connections',
    'House 2-5 Connections', 'House 2-10 Connections',
    'House 5-10 Connections', 'Royal Star Name', 'Bellatrix Star'
]

# 2. Подготовка данных
def has_aspect_func(val):
    s = str(val).lower().strip()
    return len(s) > 2 and s not in ['nan', 'none', '0', '0.0', '1', '1.0']

df_bool = df[columns_to_check].map(has_aspect_func)

# Маски для трех групп
gk_mask = df['group'] == 'goalkeeper'
ctrl_mask = df['group'] == 'control group'
outfield_mask = df['group'] == 'outfield player'

n_gk = gk_mask.sum()
n_ctrl = ctrl_mask.sum()
n_outfield = outfield_mask.sum()

# --- 3. Расчет теста Фишера для КОМБИНАЦИЙ ---
results = []
total_processed = 0

print(f"Размеры групп: Вратари={n_gk}, Контроль={n_ctrl}, Полевые={n_outfield}")

for r in range(2, 7): 
    print(f"\nАнализ комбинаций по {r} элемента(ов)...")
    for combo in combinations(columns_to_check, r):
        aspects_present = df_bool[list(combo)].all(axis=1)
        gk_with = (aspects_present & gk_mask).sum()
        
        # Проверяем только те комбинации, которые есть хотя бы у одного вратаря
        if gk_with > 0:
            gk_without = n_gk - gk_with
            
            # Тест 1: Вратари vs Контрольная группа
            ctrl_with = (aspects_present & ctrl_mask).sum()
            ctrl_without = n_ctrl - ctrl_with
            table_ctrl = [[gk_with, gk_without], [ctrl_with, ctrl_without]]
            _, p_val_ctrl = fisher_exact(table_ctrl)
            
            # Тест 2: Вратари vs Полевые игроки
            out_with = (aspects_present & outfield_mask).sum()
            out_without = n_outfield - out_with
            table_out = [[gk_with, gk_without], [out_with, out_without]]
            _, p_val_out = fisher_exact(table_out)
            
            # ФИЛЬТР: сохраняем только если ОБА теста значимы (p < 0.05)
            if p_val_ctrl < 0.05 and p_val_out < 0.05:
                results.append({
                    'Combination': " + ".join(combo),
                    'GK_count': gk_with,
                    'Ctrl_count': ctrl_with,
                    'Outfield_count': out_with,
                    'p_val_vs_Ctrl': round(p_val_ctrl, 4),
                    'p_val_vs_Outfield': round(p_val_out, 4)
                })
        
        total_processed += 1
        if total_processed % 50000 == 0:
            print(f"Проверено {total_processed} комбинаций...")

# --- 4. Вывод результатов ---
stats_df = pd.DataFrame(results)

if not stats_df.empty:
    # Сортируем по суммарной значимости
    stats_df['avg_p'] = (stats_df['p_val_vs_Ctrl'] + stats_df['p_val_vs_Outfield']) / 2
    significant = stats_df.sort_values('avg_p')

    print("\n" + "="*80)
    print("УНИКАЛЬНЫЕ КОМБИНАЦИИ ВРАТАРЕЙ")
    print("(Значимы одновременно против контроля и против полевых игроков)")
    print("="*80)
    
    # Выводим таблицу
    cols_to_show = ['Combination', 'GK_count', 'Ctrl_count', 'Outfield_count', 'p_val_vs_Ctrl', 'p_val_vs_Outfield']
    print(significant[cols_to_show].to_string(index=False))
    
    # Сохранение в файл, так как двойной фильтр — это очень ценные данные
    significant[cols_to_show].to_csv('unique_goalkeeper_signatures.csv', index=False, encoding='utf-8-sig')
    print("\nРезультаты также сохранены в файл 'unique_goalkeeper_signatures.csv'")
else:
    print("\nКомбинаций, значимых одновременно в обеих группах сравнения, не обнаружено.")

Размеры групп: Вратари=30, Контроль=30, Полевые=30

Анализ комбинаций по 2 элемента(ов)...

Анализ комбинаций по 3 элемента(ов)...

Анализ комбинаций по 4 элемента(ов)...

Анализ комбинаций по 5 элемента(ов)...
Проверено 50000 комбинаций...
Проверено 100000 комбинаций...
Проверено 150000 комбинаций...

Анализ комбинаций по 6 элемента(ов)...
Проверено 200000 комбинаций...
Проверено 250000 комбинаций...
Проверено 300000 комбинаций...
Проверено 350000 комбинаций...
Проверено 400000 комбинаций...
Проверено 450000 комбинаций...
Проверено 500000 комбинаций...
Проверено 550000 комбинаций...
Проверено 600000 комбинаций...
Проверено 650000 комбинаций...
Проверено 700000 комбинаций...
Проверено 750000 комбинаций...

Комбинаций, значимых одновременно в обеих группах сравнения, не обнаружено.


In [19]:
# Код, который найдет топ-30 комбинаций, которые максимально проявлены у вратарей и минимально — у всех остальных.
import pandas as pd
from itertools import combinations

# 1. Считываем данные
df = pd.read_csv('results.csv', encoding='utf-8-sig')

columns_to_check = [
    'Mercury-Mars', 'Mercury-Mars Dispositorship/Reception', 'Mercury-Uranus',
    'Mercury-Uranus Dispositorship/Reception', 'Mercury-Saturn',
    'Mercury-Saturn Dispositorship/Reception', 'Mercury-Pluto',
    'Mercury-Pluto Dispositorship/Reception', 'Mars-Uranus',
    'Mars-Uranus Dispositorship/Reception', 'Mars-Pluto',
    'Mars-Pluto Dispositorship/Reception', 'Mars-Moon',
    'Mars-Moon Dispositorship/Reception',
    'Mars dignity +/-', 'Mars 1st House Relation', 'Mars 2-5-10 House Relation',
    'Saturn dignity +', 'Saturn 1st House Relation',
    'Specific Signs Asc/Asc Ruler',
    'Gemini', 'Sagittarius', 'Pisces', 'Aquarius', 'House 1-10 Connections',
    'House 2-5 Connections', 'House 2-10 Connections',
    'House 5-10 Connections', 'Royal Star Name', 'Bellatrix Star'
]

# 2. Подготовка данных
def has_aspect_func(val):
    s = str(val).lower().strip()
    return len(s) > 2 and s not in ['nan', 'none', '0', '0.0', '1', '1.0']

df_bool = df[columns_to_check].map(has_aspect_func)

gk_mask = df['group'] == 'goalkeeper'
ctrl_mask = df['group'] == 'control group'
outfield_mask = df['group'] == 'outfield player'

n_gk = gk_mask.sum()
n_ctrl = ctrl_mask.sum()
n_outfield = outfield_mask.sum()

results = []
total_processed = 0

print(f"Размеры групп: Вратари={n_gk}, Контроль={n_ctrl}, Полевые={n_outfield}")

# --- 3. Поиск частотных комбинаций ---
for r in range(2, 7): 
    print(f"Анализ комбинаций по {r} элемента(ов)...")
    for combo in combinations(columns_to_check, r):
        aspects_present = df_bool[list(combo)].all(axis=1)
        
        gk_count = (aspects_present & gk_mask).sum()
        
        # Берем только те, что есть хотя бы у 3 вратарей (чтобы исключить случайные единичные совпадения)
        if gk_count >= 3:
            ctrl_count = (aspects_present & ctrl_mask).sum()
            out_count = (aspects_present & outfield_mask).sum()
            
            # Процентное содержание в каждой группе
            gk_pct = (gk_count / n_gk) * 100
            ctrl_pct = (ctrl_count / n_ctrl) * 100
            out_pct = (out_count / n_outfield) * 100
            
            # Считаем "Превосходство" (насколько у вратарей % выше, чем у остальных)
            # Прибавляем 0.1 к знаменателю, чтобы избежать деления на ноль, если в других группах 0
            diff_factor = gk_pct / (max(ctrl_pct, out_pct) + 0.1)
            
            results.append({
                'Combination': " + ".join(combo),
                'GK_%': round(gk_pct, 1),
                'Ctrl_%': round(ctrl_pct, 1),
                'Outfield_%': round(out_pct, 1),
                'GK_Absolute': gk_count,
                'Superiority_Index': round(diff_factor, 2)
            })
        
        total_processed += 1
        if total_processed % 100000 == 0:
            print(f"Проверено {total_processed} комбинаций...")

# --- 4. Формирование ТОП-30 ---
stats_df = pd.DataFrame(results)

if not stats_df.empty:
    # Сортируем по индексу превосходства и количеству карт у вратарей
    top_30 = stats_df.sort_values(by=['Superiority_Index', 'GK_Absolute'], ascending=False).head(30)

    print("\n" + "="*90)
    print("ТОП-30 СВЯЗОК: ГДЕ ВРАТАРИ ПРЕВОСХОДЯТ ОСТАЛЬНЫЕ ГРУППЫ")
    print("="*90)
    print(top_30[['Combination', 'GK_Absolute', 'GK_%', 'Ctrl_%', 'Outfield_%', 'Superiority_Index']].to_string(index=False))
    
    # Сохранение в файл
    top_30.to_csv('top_30_goalkeeper_signatures.csv', index=False, encoding='utf-8-sig')
    print("\nТоп-30 сохранен в файл 'top_30_goalkeeper_signatures.csv'")
else:
    print("\nПодходящих комбинаций не найдено.")

Размеры групп: Вратари=30, Контроль=30, Полевые=30
Анализ комбинаций по 2 элемента(ов)...
Анализ комбинаций по 3 элемента(ов)...
Анализ комбинаций по 4 элемента(ов)...
Анализ комбинаций по 5 элемента(ов)...
Проверено 100000 комбинаций...
Анализ комбинаций по 6 элемента(ов)...
Проверено 200000 комбинаций...
Проверено 300000 комбинаций...
Проверено 400000 комбинаций...
Проверено 500000 комбинаций...
Проверено 600000 комбинаций...
Проверено 700000 комбинаций...

ТОП-30 СВЯЗОК: ГДЕ ВРАТАРИ ПРЕВОСХОДЯТ ОСТАЛЬНЫЕ ГРУППЫ
                                                                                                                                                                         Combination  GK_Absolute  GK_%  Ctrl_%  Outfield_%  Superiority_Index
                                                                                                    Mercury-Uranus + Mercury-Pluto + House 1-10 Connections + House 5-10 Connections            5  16.7     0.0         0.0             166.67
  